In [ ]:
from matplotlib import pyplot as plt
from sklearn import datasets
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import math
# datasets = ['breast_cancer.csv', 'diabetes_prediction_dataset.csv']

# lr = 0.001

In [ ]:
df = pd.read_csv("5CV_MLP_102_L1to4_0_001.csv")
df = df.drop('Unnamed: 0', axis=1)
df

In [ ]:
for i in range(1, 2):    
    if i == 23 or i == 82 or i == 84:
        continue        
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i] / Select one dataset
    df_total = pd.DataFrame(df_d['L'].astype(float), columns = ['L'])
    df_total['average'] = (df_d['att_1'].astype(float) + df_d['att_2'].astype(float) + df_d['att_3'].astype(float) + df_d['att_4'].astype(float))/4
    x = df_total['L']
    y = df_total['average']
df_d

In [ ]:
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

for i in range(1, 2):    
    if i == 23 or i == 82 or i == 84:
        continue        
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i] / Select one dataset
    df_total = pd.DataFrame(df_d['L'].astype(float), columns = ['L'])
    df_total['average'] = (df_d['att_1'].astype(float) + df_d['att_2'].astype(float) + df_d['att_3'].astype(float) + df_d['att_4'].astype(float))/4
    x = df_total['L']
    y = df_total['average']

    mini = np.min(df_total['average'])
    maxi = np.max(df_total['average'])
    mean = np.mean(df_total['average'])
    lb = maxi * 0.95

#     plt.figure(figsize=(20,6))

#     plt.subplot(1, 2, 1)
    plt.plot(x,y,linewidth=3, label = 'F1-score', color = "mediumaquamarine")
    plt.axhline(lb, 0, 300, color='red', linestyle='--', linewidth=2)
    plt.title('Original Graph',fontsize = 20)
    plt.ylim(0.45, 0.70) # y축 값의 범위 설정
    plt.xlabel('L',fontsize = 20)
    plt.ylabel('Score',fontsize = 20)
    plt.legend()
    
    import matplotlib.patches as patches
    shp=patches.Ellipse((34,0.652), 10, 0.013, ec='b', fc='white', lw=3)
    plt.gca().add_patch(shp)
    plt.text(41,0.65,'The Maximum')
    
    
#     plt.subplot(1, 2, 2)
#     plt.plot(x,y,linewidth=3, label = 'F1-score')
#     plt.axhline(lb, 0, 300, color='red', linestyle='--', linewidth=2)
#     plt.title('Enlarged Graph')
#     plt.ylim(mean, maxi+0.01) # y축 값의 범위 설정
#     plt.legend()

    plt.show()

    print("threshold:", lb)
    print('first L:', df_total.loc[(df_total.average<lb) & (df_total.L > 10), ][:1].iloc[0,0])
    print(df_total[df_total['average']<lb][:5])

In [ ]:
# 오리지널 데이터 smoothing 먼저하고, 그다음 rate 계산하기
dataset = []
start_3 = []
end_3 = []
start_5 = []
end_5 = []

for i in range(1, 2):
    if i == 23 or i == 82 or i == 84:
        continue       
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i]
    df_d = df_d.astype(float)
    average = []
    for j in range(len(df_d)):
        average.append(np.mean([df_d.iloc[j,2],df_d.iloc[j,3],df_d.iloc[j,4],df_d.iloc[j,5]]))
    df_d['average'] = average 
    df_d = df_d.drop(['att_1', 'att_2', 'att_3', 'att_4'], axis=1)
    
    smooth_3 = []
    smooth_3.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2]]))
    for j in range(1, len(df_d)-1):
        smooth_3.append(np.mean([df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2]]))
    smooth_3.append(np.mean([df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_3'] = smooth_3
    
    smooth_5 = []
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2]]))
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2], df_d.iloc[3,2]]))
    for j in range(2, len(df_d)-2):
        smooth_5.append(np.mean([df_d.iloc[j-2,2], df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2], df_d.iloc[j+2,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-4,2], df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_5'] = smooth_5
    
    ave_max = np.max(average)
    ave_rate = average/ave_max
    df_d['ave_rate'] = ave_rate
    
    sm3_max = np.max(smooth_3)
    sm3_rate = smooth_3/sm3_max
    df_d['sm3_rate'] = sm3_rate
    
    sm5_max = np.max(smooth_5)
    sm5_rate = smooth_5/sm5_max
    df_d['sm5_rate'] = sm5_rate
df_d

In [ ]:
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15
# 오리지널 데이터 smoothing 먼저하고, 그다음 rate 계산하기
dataset = []
start_1 = []
end_1 = []
start_3 = []
end_3 = []
start_5 = []
end_5 = []

for i in range(1, 2):
    if i == 23 or i == 82 or i == 84:
        continue       
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i]
    df_d = df_d.astype(float)
    average = []
    for j in range(len(df_d)):
        average.append(np.mean([df_d.iloc[j,2],df_d.iloc[j,3],df_d.iloc[j,4],df_d.iloc[j,5]]))
    df_d['average'] = average 
    df_d = df_d.drop(['att_1', 'att_2', 'att_3', 'att_4'], axis=1)
    
    smooth_3 = []
    smooth_3.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2]]))
    for j in range(1, len(df_d)-1):
        smooth_3.append(np.mean([df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2]]))
    smooth_3.append(np.mean([df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_3'] = smooth_3
    
    smooth_5 = []
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2]]))
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2], df_d.iloc[3,2]]))
    for j in range(2, len(df_d)-2):
        smooth_5.append(np.mean([df_d.iloc[j-2,2], df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2], df_d.iloc[j+2,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-4,2], df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_5'] = smooth_5
    
    ave_max = np.max(average)
    ave_rate = average/ave_max
    df_d['ave_rate'] = ave_rate
    
    sm3_max = np.max(smooth_3)
    sm3_rate = smooth_3/sm3_max
    df_d['sm3_rate'] = sm3_rate
    
    sm5_max = np.max(smooth_5)
    sm5_rate = smooth_5/sm5_max
    df_d['sm5_rate'] = sm5_rate
    
#     plt.figure(figsize=(20,6))
    for k in range(2):
        if k == 0:
            plt.plot(df_d['L'],df_d['average'],linewidth=3, label = 'original', color = 'lightgray')
            plt.plot(df_d['L'],df_d['ave_rate'],linewidth=3, label = 'org_norm', color = 'lightgray')
            plt.plot(df_d['L'],df_d['smooth_3'],linewidth=3, label = 'smooth_3', color = 'deepskyblue')
            plt.plot(df_d['L'],df_d['sm3_rate'],linewidth=3, label = 'sm3_norm', color = 'royalblue')
            plt.axhline(0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.axhline(sm3_max*0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.title('Original & Smooth_3',fontsize = 20)
            plt.ylim(0, 1.0) # y축 값의 범위 설정
            plt.xlabel('L',fontsize = 20)
            plt.ylabel('Score',fontsize = 20)
            plt.legend()
            plt.show()
  
        else:
            plt.plot(df_d['L'],df_d['average'],linewidth=3, label = 'original', color = 'lightgray')
            plt.plot(df_d['L'],df_d['ave_rate'],linewidth=3, label = 'org_norm', color = 'lightgray')
            plt.plot(df_d['L'],df_d['smooth_5'],linewidth=3, label = 'smooth_5', color = 'goldenrod')
            plt.plot(df_d['L'],df_d['sm5_rate'],linewidth=3, label = 'sm5_norm', color = 'orange')
            plt.axhline(0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.axhline(sm5_max*0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.title('Original & Smooth_5',fontsize = 20)
            plt.ylim(0, 1.0) # y축 값의 범위 설정
            plt.xlabel('L',fontsize = 20)
            plt.ylabel('Score',fontsize = 20)
            plt.legend()
            plt.show()

    over_list_1 = df_d[df_d['ave_rate'] >= 0.95]["L"]    
    numlist_1 = list(range(len(over_list_1)))
    over_list_1 = pd.DataFrame(over_list_1).set_index(pd.Index(numlist_1))
    st_1 = over_list_1.iloc[0,0]
    print("START_1:", st_1)
    
    for k in range(len(over_list_1)):
        if k == len(over_list_1)-1 or over_list_1.iloc[k, 0]+3 != over_list_1.iloc[k+1, 0]:
            en_1 = over_list_1.iloc[k, 0]
            print("END_1:", en_1)
            break
            
    over_list_3 = df_d[df_d['sm3_rate'] >= 0.95]["L"]    
    numlist_3 = list(range(len(over_list_3)))
    over_list_3 = pd.DataFrame(over_list_3).set_index(pd.Index(numlist_3))
    st_3 = over_list_3.iloc[0,0]
    print("START_3:", st_3)
    
    for k in range(len(over_list_3)):
        if k == len(over_list_3)-1 or over_list_3.iloc[k, 0]+3 != over_list_3.iloc[k+1, 0]:
            en_3 = over_list_3.iloc[k, 0]
            print("END_3:", en_3)
            break
    
    over_list_5 = df_d[df_d['sm5_rate'] >= 0.95]["L"]    
    numlist_5 = list(range(len(over_list_5)))
    over_list_5 = pd.DataFrame(over_list_5).set_index(pd.Index(numlist_5))
    st_5 = over_list_5.iloc[0,0]
    print("START_5:", st_5)
    
    for k in range(len(over_list_5)):
        if k == len(over_list_5)-1 or over_list_5.iloc[k, 0]+3 != over_list_5.iloc[k+1, 0]:
            en_5 = over_list_5.iloc[k, 0]
            print("END_5:", en_5)
            break
    
    dataset.append(i)
    start_1.append(st_1)
    end_1.append(en_1)
    start_3.append(st_3)
    end_3.append(en_3)
    start_5.append(st_5)
    end_5.append(en_5)
    
res = pd.DataFrame(dataset, columns = ["dataset"])
res["start_1"] = start_1
res["end_1"] = end_1
res["start_3"] = start_3
res["end_3"] = end_3
res["start_5"] = start_5
res["end_5"] = end_5
res

In [ ]:
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15
# 오리지널 데이터 smoothing 먼저하고, 그다음 rate 계산하기
dataset = []
start_1 = []
end_1 = []
start_3 = []
end_3 = []
start_5 = []
end_5 = []

for i in range(1, 2):
    if i == 23 or i == 82 or i == 84:
        continue       
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i]
    df_d = df_d.astype(float)
    average = []
    for j in range(len(df_d)):
        average.append(np.mean([df_d.iloc[j,2],df_d.iloc[j,3],df_d.iloc[j,4],df_d.iloc[j,5]]))
    df_d['average'] = average 
    df_d = df_d.drop(['att_1', 'att_2', 'att_3', 'att_4'], axis=1)
    
    smooth_3 = []
    smooth_3.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2]]))
    for j in range(1, len(df_d)-1):
        smooth_3.append(np.mean([df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2]]))
    smooth_3.append(np.mean([df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_3'] = smooth_3
    
    smooth_5 = []
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2]]))
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2], df_d.iloc[3,2]]))
    for j in range(2, len(df_d)-2):
        smooth_5.append(np.mean([df_d.iloc[j-2,2], df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2], df_d.iloc[j+2,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-4,2], df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_5'] = smooth_5
    
    ave_max = np.max(average)
    ave_rate = average/ave_max
    df_d['ave_rate'] = ave_rate
    
    sm3_max = np.max(smooth_3)
    sm3_rate = smooth_3/sm3_max
    df_d['sm3_rate'] = sm3_rate
    
    sm5_max = np.max(smooth_5)
    sm5_rate = smooth_5/sm5_max
    df_d['sm5_rate'] = sm5_rate

    over_list_1 = df_d[df_d['ave_rate'] >= 0.95]["L"]    
    numlist_1 = list(range(len(over_list_1)))
    over_list_1 = pd.DataFrame(over_list_1).set_index(pd.Index(numlist_1))
    st_1 = over_list_1.iloc[0,0]
    print("START_1:", st_1)
    
    for k in range(len(over_list_1)):
        if k == len(over_list_1)-1 or over_list_1.iloc[k, 0]+3 != over_list_1.iloc[k+1, 0]:
            en_1 = over_list_1.iloc[k, 0]
            print("END_1:", en_1)
            break
            
    over_list_3 = df_d[df_d['sm3_rate'] >= 0.95]["L"]    
    numlist_3 = list(range(len(over_list_3)))
    over_list_3 = pd.DataFrame(over_list_3).set_index(pd.Index(numlist_3))
    st_3 = over_list_3.iloc[0,0]
    print("START_3:", st_3)
    
    for k in range(len(over_list_3)):
        if k == len(over_list_3)-1 or over_list_3.iloc[k, 0]+3 != over_list_3.iloc[k+1, 0]:
            en_3 = over_list_3.iloc[k, 0]
            print("END_3:", en_3)
            break
    
    over_list_5 = df_d[df_d['sm5_rate'] >= 0.95]["L"]    
    numlist_5 = list(range(len(over_list_5)))
    over_list_5 = pd.DataFrame(over_list_5).set_index(pd.Index(numlist_5))
    st_5 = over_list_5.iloc[0,0]
    print("START_5:", st_5)
    
    for k in range(len(over_list_5)):
        if k == len(over_list_5)-1 or over_list_5.iloc[k, 0]+3 != over_list_5.iloc[k+1, 0]:
            en_5 = over_list_5.iloc[k, 0]
            print("END_5:", en_5)
            break
    
    dataset.append(i)
    start_1.append(st_1)
    end_1.append(en_1)
    start_3.append(st_3)
    end_3.append(en_3)
    start_5.append(st_5)
    end_5.append(en_5)
    
    #     plt.figure(figsize=(20,6))
    for k in range(2):
        if k == 0:
            plt.plot(df_d['L'],df_d['average'],linewidth=3, label = 'original', color = 'lightgray')
#             plt.plot(df_d['L'],df_d['ave_rate'],linewidth=3, label = 'org_norm', color = 'lightgray')
            plt.plot(df_d['L'],df_d['smooth_3'],linewidth=3, label = 'smooth_3', color = 'royalblue')
#             plt.plot(df_d['L'],df_d['sm3_rate'],linewidth=3, label = 'sm3_norm', color = 'royalblue')
#             plt.axhline(0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.axhline(sm3_max*0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.axvline(st_3, 0.0, 0.58, color='black', linestyle=':', linewidth=3)
            plt.axvline(en_3, 0.0, 0.58, color='black', linestyle=':', linewidth=3)
            plt.title('Original & Smooth_3',fontsize = 20)
            plt.ylim(0.45, 0.70) # y축 값의 범위 설정
            plt.xlabel('L',fontsize = 20)
            plt.ylabel('Score',fontsize = 20)
            plt.legend()
            plt.show()
  
        else:
            plt.plot(df_d['L'],df_d['average'],linewidth=3, label = 'original', color = 'lightgray')
#             plt.plot(df_d['L'],df_d['ave_rate'],linewidth=3, label = 'org_norm', color = 'lightgray')
            plt.plot(df_d['L'],df_d['smooth_5'],linewidth=3, label = 'smooth_5', color = 'orange')
#             plt.plot(df_d['L'],df_d['sm5_rate'],linewidth=3, label = 'sm5_norm', color = 'orange')
#             plt.axhline(0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.axhline(sm5_max*0.95, 0, 300, color='red', linestyle='--', linewidth=2)
            plt.axvline(st_5, 0.0, 0.58, color='black', linestyle=':', linewidth=3)
            plt.axvline(en_5, 0.0, 0.58, color='black', linestyle=':', linewidth=3)
            plt.title('Original & Smooth_5',fontsize = 20)
            plt.ylim(0.45, 0.70) # y축 값의 범위 설정
            plt.xlabel('L',fontsize = 20)
            plt.ylabel('Score',fontsize = 20)
            plt.legend()
            plt.show()
    
res = pd.DataFrame(dataset, columns = ["dataset"])
res["start_1"] = start_1
res["end_1"] = end_1
res["start_3"] = start_3
res["end_3"] = end_3
res["start_5"] = start_5
res["end_5"] = end_5
res

In [ ]:
# 오리지널 데이터 smoothing 먼저하고, 그다음 rate 계산하기
dataset = []
start_1 = []
end_1 = []
start_3 = []
end_3 = []
start_5 = []
end_5 = []
maxscore_001 = []

for i in range(1, 106):
    if i == 23 or i == 82 or i == 84:
        continue       
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i]
    df_d = df_d.astype(float)
    average = []
    for j in range(len(df_d)):
        average.append(np.mean([df_d.iloc[j,2],df_d.iloc[j,3],df_d.iloc[j,4],df_d.iloc[j,5]]))
    df_d['average'] = average 
    df_d = df_d.drop(['att_1', 'att_2', 'att_3', 'att_4'], axis=1)
    
    smooth_3 = []
    smooth_3.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2]]))
    for j in range(1, len(df_d)-1):
        smooth_3.append(np.mean([df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2]]))
    smooth_3.append(np.mean([df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_3'] = smooth_3
    
    smooth_5 = []
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2]]))
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2], df_d.iloc[3,2]]))
    for j in range(2, len(df_d)-2):
        smooth_5.append(np.mean([df_d.iloc[j-2,2], df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2], df_d.iloc[j+2,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-4,2], df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_5'] = smooth_5
    
    ave_max = np.max(average)
    maxscore_001.append(ave_max)    # For each dataset, maximum score (this is for comparioson with other learning rates)
    ave_rate = average/ave_max
    df_d['ave_rate'] = ave_rate
    
    sm3_max = np.max(smooth_3)
    sm3_rate = smooth_3/sm3_max
    df_d['sm3_rate'] = sm3_rate
    
    sm5_max = np.max(smooth_5)
    sm5_rate = smooth_5/sm5_max
    df_d['sm5_rate'] = sm5_rate

    over_list_1 = df_d[df_d['ave_rate'] >= 0.95]["L"]    
    numlist_1 = list(range(len(over_list_1)))
    over_list_1 = pd.DataFrame(over_list_1).set_index(pd.Index(numlist_1))
    st_1 = over_list_1.iloc[0,0]
    print("START_1:", st_1)
    
    for k in range(len(over_list_1)):
        if k == len(over_list_1)-1 or over_list_1.iloc[k, 0]+3 != over_list_1.iloc[k+1, 0]:
            en_1 = over_list_1.iloc[k, 0]
            print("END_1:", en_1)
            break
            
    over_list_3 = df_d[df_d['sm3_rate'] >= 0.95]["L"]    
    numlist_3 = list(range(len(over_list_3)))
    over_list_3 = pd.DataFrame(over_list_3).set_index(pd.Index(numlist_3))
    st_3 = over_list_3.iloc[0,0]
    print("START_3:", st_3)
    
    for k in range(len(over_list_3)):
        if k == len(over_list_3)-1 or over_list_3.iloc[k, 0]+3 != over_list_3.iloc[k+1, 0]:
            en_3 = over_list_3.iloc[k, 0]
            print("END_3:", en_3)
            break
    
    over_list_5 = df_d[df_d['sm5_rate'] >= 0.95]["L"]    
    numlist_5 = list(range(len(over_list_5)))
    over_list_5 = pd.DataFrame(over_list_5).set_index(pd.Index(numlist_5))
    st_5 = over_list_5.iloc[0,0]
    print("START_5:", st_5)
    
    for k in range(len(over_list_5)):
        if k == len(over_list_5)-1 or over_list_5.iloc[k, 0]+3 != over_list_5.iloc[k+1, 0]:
            en_5 = over_list_5.iloc[k, 0]
            print("END_5:", en_5)
            break
    
    dataset.append(i)
    start_1.append(st_1)
    end_1.append(en_1)
    start_3.append(st_3)
    end_3.append(en_3)
    start_5.append(st_5)
    end_5.append(en_5)
    
res = pd.DataFrame(dataset, columns = ["dataset"])
res["start_1"] = start_1
res["end_1"] = end_1
res["start_3"] = start_3
res["end_3"] = end_3
res["start_5"] = start_5
res["end_5"] = end_5
res

In [ ]:
# 각 L[1, 300]에 대해서 몇 개의 데이터셋에 대해서 유효한지 
counts = pd.DataFrame(list(range(1, 301)), columns=["L"])
counts["count_1"] = 0
counts["count_3"] = 0
counts["count_5"] = 0
counts

In [ ]:
for i in range(len(res)):
    x_1 = list(range(int(list(res["start_1"])[i]), int(list(res["end_1"])[i])))
    x_3 = list(range(int(list(res["start_3"])[i]), int(list(res["end_3"])[i])))
    x_5 = list(range(int(list(res["start_5"])[i]), int(list(res["end_5"])[i])))
    for j in range(len(counts)):
        if counts["L"][j] in x_1:
            counts["count_1"][j] += 1
        if counts["L"][j] in x_3:
            counts["count_3"][j] += 1
        if counts["L"][j] in x_5:
            counts["count_5"][j] += 1
counts

In [ ]:
counts.sort_values('count_1', ascending = False)[:10]

In [ ]:
counts.sort_values('count_3', ascending = False)[:10]

In [ ]:
counts.sort_values('count_5', ascending = False)[:10]

In [ ]:
counts[14:28]

In [ ]:
# original
# plt.figure(figsize=(20,20))
for i in range(len(res)):
    x = list(range(int(list(res["start_1"])[i]), int(list(res["end_1"])[i])))
    y = [i+1 for j in range(len(x))]
    plt.plot(x,y,linewidth=1, color = "mediumaquamarine")
plt.title('95% Range (Original)')
plt.axvline(16, color='red', linestyle='--', linewidth=1)
plt.axvline(21, color='red', linestyle='--', linewidth=1)
plt.xlabel('L')
plt.ylabel('Datasets#')
plt.xlim(-1, 301) # x축 값의 범위 설정
plt.ylim(-2, 105) # y축 값의 범위 설정
plt.show()

# Smooth3
# plt.figure(figsize=(20,20))
for i in range(len(res)):
    x = list(range(int(list(res["start_3"])[i]), int(list(res["end_3"])[i])))
    y = [i+1 for j in range(len(x))]
    plt.plot(x,y,linewidth=1, color = "royalblue")
plt.title('95% Range (Smooth_3)')
plt.axvline(22, color='red', linestyle='--', linewidth=1)
plt.axvline(24, color='red', linestyle='--', linewidth=1)
plt.xlabel('L')
plt.ylabel('Datasets#')
plt.xlim(-1, 301) # x축 값의 범위 설정
plt.ylim(-2, 105) # y축 값의 범위 설정
plt.show()

# Smooth5
# plt.figure(figsize=(20,20))
for i in range(len(res)):
    x = list(range(int(list(res["start_5"])[i]), int(list(res["end_5"])[i])))
    y = [i+1 for j in range(len(x))]
    plt.plot(x,y,linewidth=1, color = "orange")
plt.title('95% Range (Smooth_5)')
plt.axvline(25, color='red', linestyle='--', linewidth=1)
plt.axvline(27, color='red', linestyle='--', linewidth=1)
plt.xlabel('L')
plt.ylabel('Datasets#')
plt.xlim(-1, 301) # x축 값의 범위 설정
plt.ylim(-2, 105) # y축 값의 범위 설정
plt.show()

In [ ]:
x = counts["L"]
y1 = counts["count_1"]
y3 = counts["count_3"]
y5 = counts["count_5"]

plt.plot(x,y1,linewidth=3, color = 'mediumaquamarine')
plt.plot(x,y3,linewidth=3, color = 'royalblue')
plt.plot(x,y5,linewidth=3, color = 'orange')
plt.title('Good Range of L (lr=0.001)')
plt.axvline(16, 0, 58/102, color='red', linestyle='--', linewidth=2)
# plt.axvline(22, 0, 81/102, color='red', linestyle='--', linewidth=2)
plt.axvline(27, 0, 91/102, color='red', linestyle='--', linewidth=2)
plt.xlabel('L')
plt.ylabel('# of Datasets')
plt.xlim(0, 301) # x축 값의 범위 설정
plt.ylim(0, 103) # y축 값의 범위 설정
plt.show()

# lr = 0.01

In [ ]:
df = pd.read_csv("5CV_MLP_102_L1to4_0_01.csv")
df = df.drop('Unnamed: 0', axis=1)
df

In [ ]:
# 오리지널 데이터 smoothing 먼저하고, 그다음 rate 계산하기
dataset = []
start_1 = []
end_1 = []
start_3 = []
end_3 = []
start_5 = []
end_5 = []
maxscore_01 = []

for i in range(1, 106):
    if i == 23 or i == 82 or i == 84:
        continue       
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i]
    df_d = df_d.astype(float)
    average = []
    for j in range(len(df_d)):
        average.append(np.mean([df_d.iloc[j,2],df_d.iloc[j,3],df_d.iloc[j,4],df_d.iloc[j,5]]))
    df_d['average'] = average 
    df_d = df_d.drop(['att_1', 'att_2', 'att_3', 'att_4'], axis=1)
    
    smooth_3 = []
    smooth_3.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2]]))
    for j in range(1, len(df_d)-1):
        smooth_3.append(np.mean([df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2]]))
    smooth_3.append(np.mean([df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_3'] = smooth_3
    
    smooth_5 = []
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2]]))
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2], df_d.iloc[3,2]]))
    for j in range(2, len(df_d)-2):
        smooth_5.append(np.mean([df_d.iloc[j-2,2], df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2], df_d.iloc[j+2,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-4,2], df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_5'] = smooth_5
    
    ave_max = np.max(average)
    maxscore_01.append(ave_max)    # For each dataset, maximum score (this is for comparioson with other learning rates)
    ave_rate = average/ave_max
    df_d['ave_rate'] = ave_rate
    
    sm3_max = np.max(smooth_3)
    sm3_rate = smooth_3/sm3_max
    df_d['sm3_rate'] = sm3_rate
    
    sm5_max = np.max(smooth_5)
    sm5_rate = smooth_5/sm5_max
    df_d['sm5_rate'] = sm5_rate

    over_list_1 = df_d[df_d['ave_rate'] >= 0.95]["L"]    
    numlist_1 = list(range(len(over_list_1)))
    over_list_1 = pd.DataFrame(over_list_1).set_index(pd.Index(numlist_1))
    st_1 = over_list_1.iloc[0,0]
    print("START_1:", st_1)
    
    for k in range(len(over_list_1)):
        if k == len(over_list_1)-1 or over_list_1.iloc[k, 0]+3 != over_list_1.iloc[k+1, 0]:
            en_1 = over_list_1.iloc[k, 0]
            print("END_1:", en_1)
            break
            
    over_list_3 = df_d[df_d['sm3_rate'] >= 0.95]["L"]    
    numlist_3 = list(range(len(over_list_3)))
    over_list_3 = pd.DataFrame(over_list_3).set_index(pd.Index(numlist_3))
    st_3 = over_list_3.iloc[0,0]
    print("START_3:", st_3)
    
    for k in range(len(over_list_3)):
        if k == len(over_list_3)-1 or over_list_3.iloc[k, 0]+3 != over_list_3.iloc[k+1, 0]:
            en_3 = over_list_3.iloc[k, 0]
            print("END_3:", en_3)
            break
    
    over_list_5 = df_d[df_d['sm5_rate'] >= 0.95]["L"]    
    numlist_5 = list(range(len(over_list_5)))
    over_list_5 = pd.DataFrame(over_list_5).set_index(pd.Index(numlist_5))
    st_5 = over_list_5.iloc[0,0]
    print("START_5:", st_5)
    
    for k in range(len(over_list_5)):
        if k == len(over_list_5)-1 or over_list_5.iloc[k, 0]+3 != over_list_5.iloc[k+1, 0]:
            en_5 = over_list_5.iloc[k, 0]
            print("END_5:", en_5)
            break
    
    dataset.append(i)
    start_1.append(st_1)
    end_1.append(en_1)
    start_3.append(st_3)
    end_3.append(en_3)
    start_5.append(st_5)
    end_5.append(en_5)
    
res = pd.DataFrame(dataset, columns = ["dataset"])
res["start_1"] = start_1
res["end_1"] = end_1
res["start_3"] = start_3
res["end_3"] = end_3
res["start_5"] = start_5
res["end_5"] = end_5
res

In [ ]:
# 각 L[1, 300]에 대해서 몇 개의 데이터셋에 대해서 유효한지 
counts = pd.DataFrame(list(range(1, 301)), columns=["L"])
counts["count_1"] = 0
counts["count_3"] = 0
counts["count_5"] = 0
for i in range(len(res)):
    x_1 = list(range(int(list(res["start_1"])[i]), int(list(res["end_1"])[i])))
    x_3 = list(range(int(list(res["start_3"])[i]), int(list(res["end_3"])[i])))
    x_5 = list(range(int(list(res["start_5"])[i]), int(list(res["end_5"])[i])))
    for j in range(len(counts)):
        if counts["L"][j] in x_1:
            counts["count_1"][j] += 1
        if counts["L"][j] in x_3:
            counts["count_3"][j] += 1
        if counts["L"][j] in x_5:
            counts["count_5"][j] += 1
counts

In [ ]:
counts.sort_values('count_1', ascending = False)[:10]

In [ ]:
counts.sort_values('count_3', ascending = False)[:10]

In [ ]:
counts.sort_values('count_5', ascending = False)[:10]

In [ ]:
x = counts["L"]
y1 = counts["count_1"]
y3 = counts["count_3"]
y5 = counts["count_5"]

plt.plot(x,y1,linewidth=3, color = 'mediumaquamarine')
plt.plot(x,y3,linewidth=3, color = 'royalblue')
plt.plot(x,y5,linewidth=3, color = 'orange')
plt.title('Good Range of L (lr=0.01)')
plt.axvline(7, 0, 79/102, color='red', linestyle='--', linewidth=2)
# plt.axvline(11, 0, 89/102, color='red', linestyle='--', linewidth=2)
plt.axvline(18, 0, 92/102, color='red', linestyle='--', linewidth=2)
plt.xlabel('L')
plt.ylabel('# of Datasets')
plt.xlim(0, 301) # x축 값의 범위 설정
plt.ylim(0, 103) # y축 값의 범위 설정
plt.show()

# lr = 0.1

In [ ]:
df = pd.read_csv("5CV_MLP_102_L1to4_0_1.csv")
df = df.drop('Unnamed: 0', axis=1)
df

In [ ]:
# 오리지널 데이터 smoothing 먼저하고, 그다음 rate 계산하기
dataset = []
start_1 = []
end_1 = []
start_3 = []
end_3 = []
start_5 = []
end_5 = []
maxscore_1 = []

for i in range(1, 106):
    if i == 23 or i == 82 or i == 84:
        continue       
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i]
    df_d = df_d.astype(float)
    average = []
    for j in range(len(df_d)):
        average.append(np.mean([df_d.iloc[j,2],df_d.iloc[j,3],df_d.iloc[j,4],df_d.iloc[j,5]]))
    df_d['average'] = average 
    df_d = df_d.drop(['att_1', 'att_2', 'att_3', 'att_4'], axis=1)
    
    smooth_3 = []
    smooth_3.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2]]))
    for j in range(1, len(df_d)-1):
        smooth_3.append(np.mean([df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2]]))
    smooth_3.append(np.mean([df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_3'] = smooth_3
    
    smooth_5 = []
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2]]))
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2], df_d.iloc[3,2]]))
    for j in range(2, len(df_d)-2):
        smooth_5.append(np.mean([df_d.iloc[j-2,2], df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2], df_d.iloc[j+2,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-4,2], df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_5'] = smooth_5
    
    ave_max = np.max(average)
    maxscore_1.append(ave_max)    # For each dataset, maximum score (this is for comparioson with other learning rates)
    ave_rate = average/ave_max
    df_d['ave_rate'] = ave_rate
    
    sm3_max = np.max(smooth_3)
    sm3_rate = smooth_3/sm3_max
    df_d['sm3_rate'] = sm3_rate
    
    sm5_max = np.max(smooth_5)
    sm5_rate = smooth_5/sm5_max
    df_d['sm5_rate'] = sm5_rate

    over_list_1 = df_d[df_d['ave_rate'] >= 0.95]["L"]    
    numlist_1 = list(range(len(over_list_1)))
    over_list_1 = pd.DataFrame(over_list_1).set_index(pd.Index(numlist_1))
    st_1 = over_list_1.iloc[0,0]
    print("START_1:", st_1)
    
    for k in range(len(over_list_1)):
        if k == len(over_list_1)-1 or over_list_1.iloc[k, 0]+3 != over_list_1.iloc[k+1, 0]:
            en_1 = over_list_1.iloc[k, 0]
            print("END_1:", en_1)
            break
            
    over_list_3 = df_d[df_d['sm3_rate'] >= 0.95]["L"]    
    numlist_3 = list(range(len(over_list_3)))
    over_list_3 = pd.DataFrame(over_list_3).set_index(pd.Index(numlist_3))
    st_3 = over_list_3.iloc[0,0]
    print("START_3:", st_3)
    
    for k in range(len(over_list_3)):
        if k == len(over_list_3)-1 or over_list_3.iloc[k, 0]+3 != over_list_3.iloc[k+1, 0]:
            en_3 = over_list_3.iloc[k, 0]
            print("END_3:", en_3)
            break
    
    over_list_5 = df_d[df_d['sm5_rate'] >= 0.95]["L"]    
    numlist_5 = list(range(len(over_list_5)))
    over_list_5 = pd.DataFrame(over_list_5).set_index(pd.Index(numlist_5))
    st_5 = over_list_5.iloc[0,0]
    print("START_5:", st_5)
    
    for k in range(len(over_list_5)):
        if k == len(over_list_5)-1 or over_list_5.iloc[k, 0]+3 != over_list_5.iloc[k+1, 0]:
            en_5 = over_list_5.iloc[k, 0]
            print("END_5:", en_5)
            break
    
    dataset.append(i)
    start_1.append(st_1)
    end_1.append(en_1)
    start_3.append(st_3)
    end_3.append(en_3)
    start_5.append(st_5)
    end_5.append(en_5)
    
res = pd.DataFrame(dataset, columns = ["dataset"])
res["start_1"] = start_1
res["end_1"] = end_1
res["start_3"] = start_3
res["end_3"] = end_3
res["start_5"] = start_5
res["end_5"] = end_5
res

In [ ]:
# 각 L[1, 300]에 대해서 몇 개의 데이터셋에 대해서 유효한지 
counts = pd.DataFrame(list(range(1, 301)), columns=["L"])
counts["count_1"] = 0
counts["count_3"] = 0
counts["count_5"] = 0
for i in range(len(res)):
    x_1 = list(range(int(list(res["start_1"])[i]), int(list(res["end_1"])[i])))
    x_3 = list(range(int(list(res["start_3"])[i]), int(list(res["end_3"])[i])))
    x_5 = list(range(int(list(res["start_5"])[i]), int(list(res["end_5"])[i])))
    for j in range(len(counts)):
        if counts["L"][j] in x_1:
            counts["count_1"][j] += 1
        if counts["L"][j] in x_3:
            counts["count_3"][j] += 1
        if counts["L"][j] in x_5:
            counts["count_5"][j] += 1
counts

In [ ]:
counts.sort_values('count_1', ascending = False)[:10]

In [ ]:
counts.sort_values('count_3', ascending = False)[:10]

In [ ]:
counts.sort_values('count_5', ascending = False)[:10]

In [ ]:
x = counts["L"]
y1 = counts["count_1"]
y3 = counts["count_3"]
y5 = counts["count_5"]

plt.plot(x,y1,linewidth=3, color = 'mediumaquamarine')
plt.plot(x,y3,linewidth=3, color = 'royalblue')
plt.plot(x,y5,linewidth=3, color = 'orange')
plt.title('Good Range of L (lr=0.1)')
plt.axvline(4, 0, 83/102, color='red', linestyle='--', linewidth=2)
# plt.axvline(7, 0, 94/102, color='red', linestyle='--', linewidth=2)
plt.axvline(9, 0, 97/102, color='red', linestyle='--', linewidth=2)
plt.xlabel('L')
plt.ylabel('# of Datasets')
plt.xlim(0, 301) # x축 값의 범위 설정
plt.ylim(0, 103) # y축 값의 범위 설정
plt.show()

###### Comparison max scores of each dataset with different learning rate
comp = pd.DataFrame(maxscore_001, columns = ["0.001"])
comp["0.01"] = maxscore_01
comp["0.1"] = maxscore_1
comp

In [ ]:
print(np.mean(maxscore_001), np.mean(maxscore_01), np.mean(maxscore_1))

# lr = 0.0001

In [ ]:
df = pd.read_csv("5CV_MLP_102_L1to4_0_0001.csv")
df = df.drop('Unnamed: 0', axis=1)
df

In [ ]:
# 오리지널 데이터 smoothing 먼저하고, 그다음 rate 계산하기
dataset = []
start_1 = []
end_1 = []
start_3 = []
end_3 = []
start_5 = []
end_5 = []
maxscore_0001 = []

for i in range(1, 106):
    if i == 23 or i == 82 or i == 84:
        continue       
    print('<{}th>'.format(i))
    df_d = df[df['Dataset#'] == '{}'.format(i)]     # df_d = df[df['Dataset#'] == i]
    df_d = df_d.astype(float)
    average = []
    for j in range(len(df_d)):
        average.append(np.mean([df_d.iloc[j,2],df_d.iloc[j,3],df_d.iloc[j,4],df_d.iloc[j,5]]))
    df_d['average'] = average 
    df_d = df_d.drop(['att_1', 'att_2', 'att_3', 'att_4'], axis=1)
    
    smooth_3 = []
    smooth_3.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2]]))
    for j in range(1, len(df_d)-1):
        smooth_3.append(np.mean([df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2]]))
    smooth_3.append(np.mean([df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_3'] = smooth_3
    
    smooth_5 = []
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2]]))
    smooth_5.append(np.mean([df_d.iloc[0,2], df_d.iloc[1,2], df_d.iloc[2,2], df_d.iloc[3,2]]))
    for j in range(2, len(df_d)-2):
        smooth_5.append(np.mean([df_d.iloc[j-2,2], df_d.iloc[j-1,2], df_d.iloc[j,2], df_d.iloc[j+1,2], df_d.iloc[j+2,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-4,2], df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    smooth_5.append(np.mean([df_d.iloc[len(df_d)-3,2], df_d.iloc[len(df_d)-2,2], df_d.iloc[len(df_d)-1,2]]))
    df_d['smooth_5'] = smooth_5
    
    ave_max = np.max(average)
    maxscore_0001.append(ave_max)    # For each dataset, maximum score (this is for comparioson with other learning rates)
    ave_rate = average/ave_max
    df_d['ave_rate'] = ave_rate
    
    sm3_max = np.max(smooth_3)
    sm3_rate = smooth_3/sm3_max
    df_d['sm3_rate'] = sm3_rate
    
    sm5_max = np.max(smooth_5)
    sm5_rate = smooth_5/sm5_max
    df_d['sm5_rate'] = sm5_rate

    over_list_1 = df_d[df_d['ave_rate'] >= 0.95]["L"]    
    numlist_1 = list(range(len(over_list_1)))
    over_list_1 = pd.DataFrame(over_list_1).set_index(pd.Index(numlist_1))
    st_1 = over_list_1.iloc[0,0]
    print("START_1:", st_1)
    
    for k in range(len(over_list_1)):
        if k == len(over_list_1)-1 or over_list_1.iloc[k, 0]+3 != over_list_1.iloc[k+1, 0]:
            en_1 = over_list_1.iloc[k, 0]
            print("END_1:", en_1)
            break
            
    over_list_3 = df_d[df_d['sm3_rate'] >= 0.95]["L"]    
    numlist_3 = list(range(len(over_list_3)))
    over_list_3 = pd.DataFrame(over_list_3).set_index(pd.Index(numlist_3))
    st_3 = over_list_3.iloc[0,0]
    print("START_3:", st_3)
    
    for k in range(len(over_list_3)):
        if k == len(over_list_3)-1 or over_list_3.iloc[k, 0]+3 != over_list_3.iloc[k+1, 0]:
            en_3 = over_list_3.iloc[k, 0]
            print("END_3:", en_3)
            break
    
    over_list_5 = df_d[df_d['sm5_rate'] >= 0.95]["L"]    
    numlist_5 = list(range(len(over_list_5)))
    over_list_5 = pd.DataFrame(over_list_5).set_index(pd.Index(numlist_5))
    st_5 = over_list_5.iloc[0,0]
    print("START_5:", st_5)
    
    for k in range(len(over_list_5)):
        if k == len(over_list_5)-1 or over_list_5.iloc[k, 0]+3 != over_list_5.iloc[k+1, 0]:
            en_5 = over_list_5.iloc[k, 0]
            print("END_5:", en_5)
            break
    
    dataset.append(i)
    start_1.append(st_1)
    end_1.append(en_1)
    start_3.append(st_3)
    end_3.append(en_3)
    start_5.append(st_5)
    end_5.append(en_5)
    
res = pd.DataFrame(dataset, columns = ["dataset"])
res["start_1"] = start_1
res["end_1"] = end_1
res["start_3"] = start_3
res["end_3"] = end_3
res["start_5"] = start_5
res["end_5"] = end_5
res

In [ ]:
print(np.mean(maxscore_0001))